# 🐾 Husky Capital Quant Desk — Lab 2: Talking to Markets with APIs
**FIN 4600 · Financial Technology Foundations · Michigan Technological University**

Welcome to your first day on the quant desk. By the end of this lab you will have:

| Mission | You will… | Badge |
|---|---|---|
| 1 | Dissect real HTTP requests and responses | 🔬 Packet Sniffer |
| 2 | Go on a **Status Code Safari** — and score your predictions | 🦁 Safari Guide |
| 3 | Build a live crypto market dashboard (CoinGecko) | 🪙 Crypto Cartographer |
| 4 | Pull stock prices straight from a Yahoo Finance **URL** and chart candlesticks | 🕯️ Candle Keeper |
| 5 | Build the **Mitten Screener** — an interactive stock screener of Michigan companies | 🔎 Screener Architect |
| 6 | Add macro context from the Federal Reserve (FRED) | 🏛️ Fed Watcher |
| 7 | Read SEC filings through EDGAR's API | 📜 Filing Hound |
| 8 | Run the **Encoding Face-Off**: JSON vs XML vs Protobuf vs Avro | 🥊 Format Fighter |

**How to work:** run cells top to bottom with **Shift + Enter**. Cells marked **🧪 Your turn** are for you to edit.
Checkpoint cells award badges automatically.

> ⚖️ **Data ethics & terms of use.** Yahoo Finance has no official public API; we call the same URL its web pages use,
> for educational use only. Be polite: cache responses, don't hammer servers, identify yourself when asked (SEC).
> Nothing in this lab is investment advice.

## 0 · Setup
If an import fails, run `pip install -r requirements.txt` in your terminal (see the README for step-by-step
Windows and macOS instructions), then restart the kernel.

In [ ]:
import io, os, sys, json, time, datetime as dt
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, clear_output

from helpers.http import fetch, show_response, clear_cache, ApiError, BROWSER_UA
from helpers.market import yahoo_history, yahoo_many, chart_json_to_frame

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.grid": True, "grid.alpha": 0.3})

print(f"Python  {sys.version.split()[0]}")
for mod in (pd, np, requests):
    print(f"{mod.__name__:<8}{mod.__version__}")

### 🧪 Your turn — tell the APIs who you are
The SEC **requires** automated clients to identify themselves with a name and email in the `User-Agent` header.
API keys are optional for this lab. If you get free keys, put them in a file named `.env`
(copy `.env.example`) — **never paste keys into a notebook you push to GitHub.**

In [ ]:
YOUR_NAME  = "First Last"          # TODO: your name
YOUR_EMAIL = "your_uid@mtu.edu"    # TODO: your MTU email address

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

COINGECKO_DEMO_KEY   = os.getenv("COINGECKO_DEMO_KEY", "")     # optional, free
ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY", "demo")  # 'demo' works for IBM only
FRED_API_KEY         = os.getenv("FRED_API_KEY", "")           # optional, free
SEC_UA = f"{YOUR_NAME} {YOUR_EMAIL}"

BADGES, TOTAL_BADGES = {}, 8
def checkpoint(badge, ok, hint=""):
    BADGES[badge] = bool(ok)
    print(("✅ " if ok else "❌ ") + badge + ("" if ok else f"  — hint: {hint}"))
    print(f"🏅 Badges earned: {sum(BADGES.values())}/{TOTAL_BADGES}")

print("SEC User-Agent:", SEC_UA)
print("Keys loaded   :", {"CoinGecko": bool(COINGECKO_DEMO_KEY),
                          "AlphaVantage": ALPHAVANTAGE_API_KEY != "demo", "FRED": bool(FRED_API_KEY)})
if "your_uid" in YOUR_EMAIL:
    print("⚠️  Update YOUR_NAME and YOUR_EMAIL above before Mission 7.")

---
## Mission 1 · 🔬 Anatomy of an HTTP request
### Warm-up: the same call you made with `curl` in lecture
Lines that start with `!` run in your system shell. (On Windows, Jupyter uses `cmd.exe`, where `curl` really is
`curl.exe` — PowerShell's `curl` alias doesn't apply here.)

In [ ]:
!curl -s "https://api.frankfurter.dev/v1/latest?base=USD&symbols=EUR,CAD,JPY"

Now the same request from Python with the `requests` library. A **URL** has parts:

```
https://api.frankfurter.dev/v1/latest?base=USD&symbols=EUR,CAD,JPY
└─┬─┘   └────────┬────────┘└────┬────┘└──────────────┬───────────┘
scheme        host            path          query string (key=value pairs joined by &)
```

In [ ]:
r = requests.get("https://api.frankfurter.dev/v1/latest",
                 params={"base": "USD", "symbols": "EUR,CAD,JPY"},
                 timeout=20)
show_response(r)

In [ ]:
fx = r.json()          # JSON text → Python dict
print(type(fx))
print(json.dumps(fx, indent=2))

`requests` built the query string for us — including **URL encoding** of characters like spaces and commas.
Let's see exactly what goes on the wire without sending anything:

In [ ]:
prepared = requests.Request("GET", "https://example.com/search",
                            params={"q": "Ford Motor & Co", "symbols": "F,GM"}).prepare()
print(prepared.url)

### 🧪 Your turn — time travel
Frankfurter serves historical rates at `https://api.frankfurter.dev/v1/{YYYY-MM-DD}`.
Build a request for **15 September 2008** (the day Lehman Brothers filed for bankruptcy), base **USD**,
symbols **EUR, JPY, CHF**. How many yen did a dollar buy that day?

In [ ]:
lehman_url    = None   # TODO: e.g. "https://api.frankfurter.dev/v1/...."
lehman_params = {}     # TODO: {"base": ..., "symbols": ...}

if lehman_url:
    lehman = requests.get(lehman_url, params=lehman_params, timeout=20).json()
    print(lehman)

In [ ]:
checkpoint("🔬 Packet Sniffer",
           r.status_code == 200 and "rates" in fx and "lehman" in globals(),
           "run the Frankfurter cells and complete the time-travel exercise")

---
## Mission 2 · 🦁 Status Code Safari
Every HTTP response starts with a three-digit **status code**:
`1xx` informational · `2xx` success · `3xx` redirection · `4xx` *you* made a mistake · `5xx` *the server* made a mistake.

Below are ten real requests. **Before running anything, predict each status code** in the next cell.
Then run the safari and see how you score. Hints: think about missing identification, retired endpoints,
things that don't exist, and APIs that report errors *inside* a 200 response.

In [ ]:
CHART = "https://query1.finance.yahoo.com/v8/finance/chart/F?range=5d&interval=1d"
SEC_URL = "https://data.sec.gov/submissions/CIK0000037996.json"   # Ford Motor Co.

safari = [
  # (description, url, headers)
  ("Frankfurter · valid request",                    "https://api.frankfurter.dev/v1/latest?base=USD&symbols=EUR", {}),
  ("Frankfurter · made-up currency XYZ",             "https://api.frankfurter.dev/v1/latest?base=XYZ", {}),
  ("CoinGecko · a coin that doesn't exist",          "https://api.coingecko.com/api/v3/coins/not-a-real-coin", {}),
  ("SEC EDGAR · no contact info in User-Agent",      SEC_URL, {}),
  ("SEC EDGAR · with your name + email",             SEC_URL, {"User-Agent": SEC_UA}),
  ("Yahoo · v7 quote endpoint (retired for anonymous use)",
                                                     "https://query1.finance.yahoo.com/v7/finance/quote?symbols=F",
                                                     {"User-Agent": BROWSER_UA}),
  ("Yahoo · v8 chart with Python's default User-Agent", CHART, {}),
  ("Yahoo · v8 chart with a browser User-Agent",     CHART, {"User-Agent": BROWSER_UA}),
  ("Alpha Vantage · 'demo' key on a non-demo symbol",
                                                     "https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol=F&apikey=demo", {}),
  ("httpbin · ask the server to be a teapot",        "https://httpbin.org/status/418", {}),
]
for i, (desc, _, _) in enumerate(safari, 1):
    print(f"{i:>2}. {desc}")

In [ ]:
# 🧪 Your turn: predict a status code for each experiment (e.g. 200, 401, 403, 404, 418, 429)
my_predictions = {1: None, 2: None, 3: None, 4: None, 5: None,
                  6: None, 7: None, 8: None, 9: None, 10: None}

In [ ]:
LIAR_WORDS = ("Information", "Error Message", "Note")   # Alpha Vantage puts errors in a 200 body

rows = []
for i, (desc, url, headers) in enumerate(safari, 1):
    try:
        resp = requests.get(url, headers=headers, timeout=20)
        body = resp.text[:200]
        status, reason = resp.status_code, resp.reason
    except requests.RequestException as exc:
        status, reason, body = None, type(exc).__name__, str(exc)[:200]
    lying = status == 200 and any(f'"{w}' in body for w in LIAR_WORDS)
    rows.append({"#": i, "experiment": desc, "predicted": my_predictions.get(i),
                 "actual": status, "reason": reason,
                 "note": "🤥 200 with an error inside!" if lying else "",
                 "body preview": body.replace("\n", " ")[:70]})
    time.sleep(0.5)   # be polite

safari_df = pd.DataFrame(rows).set_index("#")
score = int((safari_df.predicted == safari_df.actual).sum())

def color_status(v):
    if v is None or pd.isna(v): return ""
    return {2: "background-color:#d4f4dd", 3: "background-color:#dbeafe",
            4: "background-color:#fde2cf", 5: "background-color:#f8d0d0"}.get(int(v) // 100, "")

display(safari_df.style.map(color_status, subset=["actual"]))
print(f"\n🎯 You predicted {score}/10 status codes correctly.")

**Debrief questions** (answer in the markdown cell below):
1. Which responses would a robust program **retry** (with back-off), and which should it **never** retry?
2. Experiment 9 returned **200 OK** but the data is missing. Why is that dangerous for an automated trading or
   reporting system, and how would you detect it?
3. Experiments 7 and 8 hit the *same URL*. What changed? What does that tell you about how Yahoo protects its endpoint?
4. Your result for 7 may differ from a classmate's. Why might that happen on a shared campus network?

✍️ *Your answers here…*

In [ ]:
checkpoint("🦁 Safari Guide", safari_df.actual.notna().sum() >= 6 and score >= 1,
           "fill in my_predictions and run the safari")

---
## Mission 3 · 🪙 Crypto Market Pulse (CoinGecko)
CoinGecko offers a **keyless** public API, but its rate limit is shared by everyone on the same IP address —
i.e., the whole classroom. If you see `429 Too Many Requests`, wait a minute or get a free **Demo key**
(sent in the `x-cg-demo-api-key` header). Our `fetch()` helper caches results and backs off automatically.

In [ ]:
cg_headers = {"x-cg-demo-api-key": COINGECKO_DEMO_KEY} if COINGECKO_DEMO_KEY else {}
try:
    markets = fetch("https://api.coingecko.com/api/v3/coins/markets",
                    params={"vs_currency": "usd", "order": "market_cap_desc", "per_page": 25,
                            "page": 1, "price_change_percentage": "24h,7d"},
                    headers=cg_headers, ttl=300)
    coins = pd.DataFrame(markets)
    print(f"{len(coins)} coins, {coins.shape[1]} columns")
except (ApiError, requests.RequestException) as exc:
    coins = None
    print("CoinGecko unavailable right now:", exc)

if coins is not None:
    display(coins[["name", "symbol", "current_price", "market_cap", "total_volume",
                   "price_change_percentage_24h_in_currency",
                   "price_change_percentage_7d_in_currency"]].head(10))

In [ ]:
if coins is not None:
    fig = px.treemap(coins, path=["name"], values="market_cap",
                     color="price_change_percentage_24h_in_currency",
                     color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
                     title="Top 25 crypto assets — size = market cap, color = 24h % change")
    fig.update_layout(height=500, margin=dict(t=50, l=10, r=10, b=10))
    fig.show()

### 🧪 Your turn — who moved the most this week?
Switch the chart below from the 24-hour change to the **7-day** change, then answer: which coin was the
biggest winner and which the biggest loser? Are stablecoins (USDT, USDC…) near zero as expected?

In [ ]:
change_col = "price_change_percentage_24h_in_currency"   # TODO: switch to the 7-day column

In [ ]:
if coins is not None:
    s = coins.set_index("name")[change_col].dropna().sort_values()
    colors = np.where(s >= 0, "#2e9e5b", "#d2483a")
    ax = s.plot.barh(color=colors, figsize=(9, 7), title=f"{change_col.replace('_', ' ')} (%)")
    ax.axvline(0, color="black", lw=0.8)
    plt.tight_layout(); plt.show()
    print("Biggest winner:", s.idxmax(), f"{s.max():+.2f}%")
    print("Biggest loser :", s.idxmin(), f"{s.min():+.2f}%")

In [ ]:
checkpoint("🪙 Crypto Cartographer", coins is not None and "7d" in change_col,
           "load CoinGecko data and switch change_col to the 7-day column")

---
## Mission 4 · 🕯️ Yahoo Finance at the URL level
Yahoo's web pages load price history from this URL pattern (no key required, but a browser `User-Agent` is):

```
https://query1.finance.yahoo.com/v8/finance/chart/{SYMBOL}?range=1y&interval=1d
```
| Parameter | Examples |
|---|---|
| `range` | `1d 5d 1mo 3mo 6mo 1y 2y 5y 10y ytd max` |
| `interval` | `1m 5m 15m 1h 1d 1wk 1mo` (1-minute bars only go back ~7 days) |

In [ ]:
symbol = "F"   # Ford Motor Company — Dearborn, Michigan
url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
payload = fetch(url, params={"range": "1y", "interval": "1d"},
                headers={"User-Agent": BROWSER_UA}, ttl=3600)

result = payload["chart"]["result"][0]
print("Top-level keys :", list(payload))
print("result keys    :", list(result))
print("indicator keys :", list(result["indicators"]))
meta = result["meta"]
print(f"\n{meta['symbol']} on {meta.get('fullExchangeName', meta.get('exchangeName'))}: "
      f"last {meta.get('regularMarketPrice')} {meta.get('currency')}")

The JSON is **columnar**: one list of timestamps and parallel lists of open/high/low/close/volume.
That maps naturally onto a pandas DataFrame:

In [ ]:
quote = result["indicators"]["quote"][0]
f_df = pd.DataFrame(quote, index=pd.to_datetime(result["timestamp"], unit="s", utc=True)
                                  .tz_convert(meta.get("exchangeTimezoneName", "America/New_York")))
f_df.index.name = "date"
f_df = f_df[["open", "high", "low", "close", "volume"]].dropna()
display(f_df.tail())
display(f_df.describe().round(2))

In [ ]:
def candle_chart(df, title):
    d = df.copy()
    d["SMA50"], d["SMA200"] = d.close.rolling(50).mean(), d.close.rolling(200).mean()
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75, 0.25], vertical_spacing=0.03)
    fig.add_trace(go.Candlestick(x=d.index, open=d.open, high=d.high, low=d.low, close=d.close,
                                 name="Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=d.index, y=d.SMA50, name="50-day SMA", line=dict(width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=d.index, y=d.SMA200, name="200-day SMA", line=dict(width=1.5)), row=1, col=1)
    fig.add_trace(go.Bar(x=d.index, y=d.volume, name="Volume", marker_color="#999"), row=2, col=1)
    fig.update_layout(title=title, height=560, xaxis_rangeslider_visible=False,
                      legend=dict(orientation="h", y=1.02, x=0))
    fig.show()

candle_chart(f_df, f"{symbol} — daily candles, 1 year")

### 🧪 Your turn — your own chart
Pick any ticker (e.g. `GM`, `NVDA`, `^GSPC` for the S&P 500, `BTC-USD`, `EURUSD=X`) and a range/interval.
Our helper `yahoo_history()` wraps exactly the URL call above, plus caching and error handling.

In [ ]:
my_symbol, my_range, my_interval = "GM", "6mo", "1d"   # TODO: change these

In [ ]:
mine = yahoo_history(my_symbol, range_=my_range, interval=my_interval, verbose=True)
candle_chart(mine, f"{my_symbol} — {my_range} @ {my_interval}")

**Library vs. URL.** Popular libraries such as `yfinance` wrap these same URLs. Uncomment to compare
(the library may be slower and can break when Yahoo changes things — you now know why).
```python
# import yfinance as yf
# yf.download("F", period="1y", auto_adjust=True).tail()
```

In [ ]:
checkpoint("🕯️ Candle Keeper", len(f_df) > 100 and len(mine) > 0,
           "load Ford's history and chart your own ticker")

---
## Mission 5 · 🔎 The Mitten Screener
A **stock screener** filters a universe of securities by rules — the first tool on every analyst's desk.
Our universe: public companies headquartered in **Michigan**, plus `SPY` (S&P 500 ETF) as a benchmark.

> Tickers change: companies merge, get acquired, or delist. If one fails below, that's real-world data
> engineering — note it and move on (or fix the list!).

In [ ]:
UNIVERSE = {
    "F": ("Ford Motor", "Dearborn"),            "GM": ("General Motors", "Detroit"),
    "SYK": ("Stryker", "Portage"),              "DOW": ("Dow Inc.", "Midland"),
    "WHR": ("Whirlpool", "Benton Harbor"),      "DTE": ("DTE Energy", "Detroit"),
    "CMS": ("CMS Energy", "Jackson"),           "ALLY": ("Ally Financial", "Detroit"),
    "DPZ": ("Domino's Pizza", "Ann Arbor"),     "MAS": ("Masco", "Livonia"),
    "LEA": ("Lear", "Southfield"),              "BWA": ("BorgWarner", "Auburn Hills"),
    "GNTX": ("Gentex", "Zeeland"),              "PAG": ("Penske Automotive", "Bloomfield Hills"),
    "RKT": ("Rocket Companies", "Detroit"),     "CACC": ("Credit Acceptance", "Southfield"),
    "UFPI": ("UFP Industries", "Grand Rapids"), "MLKN": ("MillerKnoll", "Zeeland"),
    "WWW": ("Wolverine World Wide", "Rockford"),
    "SPY": ("S&P 500 ETF (benchmark)", "—"),
}
histories = yahoo_many(list(UNIVERSE), range_="1y")
n_synth = sum(df.attrs.get("synthetic", False) for df in histories.values())
if n_synth:
    print(f"\n⚠️  {n_synth} symbols are using SYNTHETIC data — results are for practice only.")

### Risk-free rate from the Fed
Sharpe ratios need a risk-free rate. FRED publishes the 3-month Treasury bill yield (series `DGS3MO`)
as a keyless CSV download.

In [ ]:
def fred_series(series_id):
    csv_text = fetch("https://fred.stlouisfed.org/graph/fredgraph.csv",
                     params={"id": series_id}, kind="text", ttl=6 * 3600)
    s = pd.read_csv(io.StringIO(csv_text))
    s.columns = ["date", series_id]                 # header name changed over time; rename by position
    s["date"] = pd.to_datetime(s["date"])
    s[series_id] = pd.to_numeric(s[series_id], errors="coerce")   # FRED uses "." for missing values
    return s.dropna().set_index("date")[series_id]

try:
    RF = fred_series("DGS3MO").iloc[-1] / 100
    print(f"3-month T-bill yield: {RF:.2%}")
except Exception as exc:
    RF = 0.0
    print("FRED unavailable, using RF = 0:", exc)

### The metrics
| Metric | Formula / idea | Why analysts care |
|---|---|---|
| 1M / 3M / 6M / 12M return | $P_t / P_{t-n} - 1$ | Momentum |
| Annualized volatility | $\sigma_{daily}\sqrt{252}$ | Risk |
| Sharpe ratio | $(R_{ann} - r_f)/\sigma_{ann}$ | Return per unit of risk |
| Max drawdown | worst peak-to-trough fall | Pain tolerance |
| % off 52-week high | $P_t / \max(P) - 1$ | Value / mean-reversion |
| RSI(14) | Wilder's relative strength index, 0–100 | Overbought (>70) / oversold (<30) |
| Trend | price vs. 200-day SMA; 50-day vs. 200-day ("golden cross") | Technical trend filter |

In [ ]:
def rsi(close, n=14):
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + gain / loss)

def trailing_return(px_, days):
    return px_.iloc[-1] / px_.iloc[-days - 1] - 1 if len(px_) > days else np.nan

def metrics(sym, df, rf=RF):
    p = df["adj_close"].dropna()
    r = p.pct_change().dropna()
    ann_ret = (p.iloc[-1] / p.iloc[0]) ** (252 / max(len(r), 1)) - 1
    vol = r.std() * np.sqrt(252)
    sma50, sma200 = p.rolling(50).mean().iloc[-1], p.rolling(200).mean().iloc[-1]
    return {
        "symbol": sym, "name": UNIVERSE.get(sym, (sym, ""))[0], "hq": UNIVERSE.get(sym, ("", ""))[1],
        "price": p.iloc[-1],
        "ret_1m": trailing_return(p, 21), "ret_3m": trailing_return(p, 63),
        "ret_6m": trailing_return(p, 126), "ret_12m": trailing_return(p, 250),
        "vol": vol, "sharpe": (ann_ret - rf) / vol if vol else np.nan,
        "max_dd": (p / p.cummax() - 1).min(), "off_high": p.iloc[-1] / p.max() - 1,
        "rsi14": rsi(p).iloc[-1],
        "above_200d": bool(p.iloc[-1] > sma200) if not np.isnan(sma200) else False,
        "golden_cross": bool(sma50 > sma200) if not np.isnan(sma200) else False,
        "avg_dollar_vol_m": (df["close"] * df["volume"]).tail(63).mean() / 1e6,
        "synthetic": df.attrs.get("synthetic", False),
    }

scr = pd.DataFrame([metrics(s, d) for s, d in histories.items() if len(d) > 30]).set_index("symbol")
pct_cols = ["ret_1m", "ret_3m", "ret_6m", "ret_12m", "vol", "max_dd", "off_high"]
display(scr.sort_values("sharpe", ascending=False)
           .style.format({c: "{:+.1%}" for c in pct_cols} | {"price": "{:.2f}", "sharpe": "{:.2f}",
                                                             "rsi14": "{:.0f}", "avg_dollar_vol_m": "{:,.0f}"})
           .background_gradient(subset=["ret_3m", "sharpe"], cmap="RdYlGn")
           .background_gradient(subset=["vol"], cmap="Reds"))

### Interactive screener
Move the sliders — the table and chart update live. (If sliders don't appear, your Jupyter front-end lacks
widget support; the fallback cell below works everywhere.)

In [ ]:
def apply_screen(m, min_ret_3m=-1.0, max_vol=2.0, worst_dd=-1.0, need_trend=False, rsi_range=(0, 100)):
    out = m[(m.ret_3m >= min_ret_3m) & (m.vol <= max_vol) & (m.max_dd >= worst_dd)
            & m.rsi14.between(*rsi_range)]
    if need_trend:
        out = out[out.above_200d]
    return out.sort_values("sharpe", ascending=False)

def risk_return_chart(m, title="Risk vs. return (bubble = avg daily $ volume)"):
    d = m.reset_index()
    fig = px.scatter(d, x="vol", y="ret_12m", size="avg_dollar_vol_m", color="sharpe", text="symbol",
                     hover_name="name", color_continuous_scale="RdYlGn", size_max=45,
                     labels={"vol": "Annualized volatility", "ret_12m": "12-month return"}, title=title)
    fig.update_traces(textposition="top center")
    fig.update_layout(height=520, xaxis_tickformat=".0%", yaxis_tickformat=".0%")
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.show()

SHOW = ["name", "hq", "price", "ret_3m", "ret_12m", "vol", "sharpe", "max_dd", "rsi14", "above_200d"]
FMT = {"ret_3m": "{:+.1%}", "ret_12m": "{:+.1%}", "vol": "{:.1%}", "max_dd": "{:.1%}",
       "price": "{:.2f}", "sharpe": "{:.2f}", "rsi14": "{:.0f}"}

try:
    import ipywidgets as W
    w_ret = W.IntSlider(value=-20, min=-50, max=50, step=5, description="Min 3M %")
    w_vol = W.IntSlider(value=60, min=10, max=100, step=5, description="Max vol %")
    w_dd = W.IntSlider(value=-50, min=-80, max=0, step=5, description="Worst DD %")
    w_rsi = W.IntRangeSlider(value=[0, 100], min=0, max=100, step=5, description="RSI range")
    w_trend = W.Checkbox(value=False, description="Above 200-day SMA only")
    out = W.Output()

    def refresh(_=None):
        res = apply_screen(scr, w_ret.value / 100, w_vol.value / 100, w_dd.value / 100,
                           w_trend.value, tuple(w_rsi.value))
        with out:
            clear_output(wait=True)
            print(f"{len(res)} of {len(scr)} pass the screen")
            display(res[SHOW].style.format(FMT))
            if len(res):
                risk_return_chart(res, "Screen survivors")

    for w_ in (w_ret, w_vol, w_dd, w_rsi, w_trend):
        w_.observe(refresh, names="value")
    display(W.VBox([W.HBox([w_ret, w_vol]), W.HBox([w_dd, w_rsi]), w_trend, out]))
    refresh()
except ImportError:
    print("ipywidgets not installed — use the static screen below.")

In [ ]:
# Static version (works everywhere): the whole universe on one chart
risk_return_chart(scr)

### The horse race 🏇
Rebase every price to 100 at the start and watch the top momentum names race the S&P 500.

In [ ]:
leaders = scr.drop(index="SPY", errors="ignore").nlargest(5, "ret_6m").index.tolist() + ["SPY"]
race = pd.DataFrame({s: histories[s]["adj_close"] for s in leaders}).dropna()
race.index = race.index.tz_localize(None).normalize()
race = race / race.iloc[0] * 100
ax = race.plot(figsize=(10, 5), title="Growth of $100 — top 6-month momentum vs. SPY")
ax.axhline(100, color="black", lw=0.7)
ax.set_ylabel("Value of $100")
plt.tight_layout(); plt.show()

In [ ]:
rets = pd.DataFrame({s: d["adj_close"] for s, d in histories.items()})
rets.index = pd.to_datetime(rets.index).tz_localize(None).normalize()
corr = rets.pct_change().dropna(how="all").corr()
fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, text_auto=".1f",
                title="Correlation of daily returns — who moves together?")
fig.update_layout(height=650)
fig.show()

### 🧪 Your turn — design your own screen
Write a function that returns the stocks passing **your** rules. Example idea — *quality momentum*:
positive 6-month return, volatility below 35%, price above the 200-day SMA, and RSI below 70 (not overbought).

In [ ]:
def my_screen(m):
    # TODO: replace this line with your own rules, e.g. m[(m.ret_6m > 0) & (m.vol < 0.35)]
    return m

my_picks = my_screen(scr)
display(my_picks[SHOW].style.format(FMT))

### 🧪 Your turn — add a metric: the Sortino ratio
Sharpe penalizes *all* volatility. **Sortino** only penalizes downside volatility:
$$\text{Sortino} = \frac{R_{ann} - r_f}{\sigma_{down}\sqrt{252}}, \qquad \sigma_{down} = \text{std of daily returns that are } < 0$$

In [ ]:
def sortino(prices, rf=RF):
    # TODO: compute daily returns, annualized return, downside deviation, and the ratio
    return np.nan

scr["sortino"] = [sortino(histories[s]["adj_close"]) for s in scr.index]
display(scr[["name", "sharpe", "sortino"]].sort_values("sortino", ascending=False).round(2))

> 🏆 **Class challenge — the Mitten Cup.** Each team submits its top 3 picks from its own screen.
> We'll re-run this notebook at the end of the semester and compare equal-weighted returns against SPY.
> (Bragging rights only — this is an exercise, not investment advice.)

In [ ]:
checkpoint("🔎 Screener Architect",
           len(scr) >= 10 and "sortino" in scr and scr["sortino"].notna().any() and len(my_picks) < len(scr),
           "implement my_screen (it must filter something) and sortino()")

---
## Mission 6 · 🏛️ Macro context from FRED
The 10-year minus 2-year Treasury spread is a famous recession signal — when it goes negative
("yield-curve inversion"), markets take notice. We reuse `fred_series()` from Mission 5.

In [ ]:
try:
    ten, two = fred_series("DGS10"), fred_series("DGS2")
    curve = pd.concat([ten, two], axis=1).dropna().loc["2000":]
    curve["spread"] = curve.DGS10 - curve.DGS2
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(curve.index, curve.spread, lw=1, color="#1f3b73")
    ax.fill_between(curve.index, curve.spread, 0, where=curve.spread < 0, color="#d2483a", alpha=0.4,
                    label="Inverted (10y < 2y)")
    ax.axhline(0, color="black", lw=0.8)
    ax.set(title="US Treasury 10y − 2y spread (percentage points)", ylabel="pp")
    ax.legend(); plt.tight_layout(); plt.show()
except Exception as exc:
    curve = None
    print("FRED unavailable:", exc)

### 🧪 Your turn — pick a series
Browse [fred.stlouisfed.org](https://fred.stlouisfed.org), find a series, and copy its ID from the URL
(e.g. `UNRATE` unemployment, `CPIAUCSL` CPI, `MORTGAGE30US` 30-year mortgage rate, `DCOILWTICO` WTI oil).

*Optional:* with a free FRED API key in `.env`, the official JSON API is
`https://api.stlouisfed.org/fred/series/observations?series_id=UNRATE&api_key=KEY&file_type=json`.

In [ ]:
my_series = "UNRATE"   # TODO: choose another FRED series ID

In [ ]:
try:
    s = fred_series(my_series).loc["2000":]
    s.plot(title=f"FRED: {my_series}", figsize=(10, 4)); plt.tight_layout(); plt.show()
    if FRED_API_KEY:
        official = fetch("https://api.stlouisfed.org/fred/series/observations",
                         params={"series_id": my_series, "api_key": FRED_API_KEY,
                                 "file_type": "json", "sort_order": "desc", "limit": 3})
        print(json.dumps(official["observations"], indent=1))
except Exception as exc:
    print("FRED unavailable:", exc)

In [ ]:
checkpoint("🏛️ Fed Watcher", curve is not None and my_series != "UNRATE",
           "plot the yield-curve spread and choose your own series")

---
## Mission 7 · 📜 SEC EDGAR — filings as data
EDGAR's APIs are free and keyless, but the SEC's fair-access policy requires a descriptive `User-Agent`
(name + email) and at most **10 requests per second**. You met the `403` in the safari.
Companies are identified by a 10-digit **CIK** (Central Index Key). Ford's is `0000037996`.

In [ ]:
try:
    sub = fetch("https://data.sec.gov/submissions/CIK0000037996.json",
                headers={"User-Agent": SEC_UA}, ttl=3600)
    print(sub["name"], "|", sub.get("sicDescription"), "|", sub.get("tickers"))
    filings = pd.DataFrame(sub["filings"]["recent"])
    filings["filingDate"] = pd.to_datetime(filings["filingDate"])
    display(filings[["filingDate", "form", "primaryDocDescription", "accessionNumber"]].head(10))
    counts = filings[filings.filingDate >= filings.filingDate.max() - pd.Timedelta(days=365)].form.value_counts().head(10)
    counts.sort_values().plot.barh(title="Ford: filings by form type, last 12 months", figsize=(8, 4))
    plt.tight_layout(); plt.show()
except Exception as exc:
    filings = None
    print("EDGAR request failed:", exc)

**What are these forms?** `10-K` annual report · `10-Q` quarterly · `8-K` material event ·
`4` insider transaction · `DEF 14A` proxy statement · `424B2` prospectus supplement (debt offerings).
Structured financial statement data (XBRL) lives at `https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json`.

In [ ]:
checkpoint("📜 Filing Hound", filings is not None and "your_uid" not in SEC_UA,
           "set YOUR_NAME / YOUR_EMAIL in Setup and re-run")

---
## Mission 8 · 🥊 The Encoding Face-Off — contenders to JSON's throne
JSON is the default language of web APIs, but it isn't the only one. Let's encode one **live** Ford quote
six different ways and compare.

In [ ]:
from helpers.encodings import encode_all, hexdump, PROTO_TEXT, AVRO_SCHEMA

live_quote = {
    "symbol": meta["symbol"], "currency": meta.get("currency", "USD"),
    "price": float(meta.get("regularMarketPrice", f_df.close.iloc[-1])),
    "volume": int(meta.get("regularMarketVolume", f_df.volume.iloc[-1])),
    "time_unix": int(meta.get("regularMarketTime", time.time())),
}
encoded = encode_all(live_quote)
print("XML :", encoded["XML"].decode(), "\n")
print("JSON:", encoded["JSON"].decode(), "\n")
print("Protocol Buffers schema (.proto):\n" + PROTO_TEXT, "\n")
print("Protocol Buffers bytes on the wire:\n" + hexdump(encoded["Protocol Buffers"]))

In [ ]:
sizes = pd.Series({k: len(v) for k, v in encoded.items()}).sort_values()
ax = sizes.plot.barh(color=["#2e9e5b" if v < 40 else "#f0b400" if v < 90 else "#d2483a" for v in sizes],
                     title="Bytes to send one quote", figsize=(8, 3.8))
for i, v in enumerate(sizes):
    ax.text(v + 1, i, f"{v} B", va="center")
ax.set_xlabel("bytes"); plt.tight_layout(); plt.show()
print(f"JSON is {sizes['JSON'] / sizes['Protocol Buffers']:.1f}× larger than Protobuf for this message.")

**Why the difference?** Text formats repeat every field *name* in every message and write numbers as
characters. Protobuf and Avro move the names into a **schema** both sides already share; on the wire,
Protobuf sends a tiny field *number* + a binary value, and Avro sends only the values, in schema order.

### 🧪 Your turn — schema-first vs. schemaless
Add a new field (`exchange`) to the quote. Which formats accept it without any change, and which refuse
because it isn't in their schema? What does that mean for two companies exchanging messages?

In [ ]:
extended = live_quote | {"exchange": meta.get("exchangeName", "NYQ")}
from helpers.encodings import ENCODERS
for name, fn in ENCODERS.items():
    try:
        b = fn(extended)
        print(f"✅ {name:<17} accepted ({len(b)} bytes)")
    except Exception as exc:
        print(f"❌ {name:<17} rejected: {type(exc).__name__}: {str(exc)[:60]}")

✍️ *Your observations…* (Hint: Avro silently ignored the extra field — is that good or bad? Protobuf
refused it. How do schema registries and versioned `.proto` files help large organizations — or standards
bodies such as ISO 20022 and FIX — evolve messages safely?)

In [ ]:
checkpoint("🥊 Format Fighter", len(encoded) >= 5, "install protobuf, fastavro, msgpack and cbor2")

---
## 🎓 Debrief & submission
1. Which API was easiest to use and which was hardest? Why (authentication, documentation, format, limits)?
2. Give one example where a `200 OK` should **not** be trusted, and how you'd guard against it.
3. You're designing an internal market-data service for a trading firm. Would you choose REST+JSON,
   GraphQL, or gRPC+Protobuf? Justify in 3–4 sentences.
4. What ethical and legal responsibilities come with using an *unofficial* API like Yahoo's in a product?

✍️ *Your answers here…*

**Submit:** File → *Save and Export Notebook As…* → **HTML**, and upload to Canvas.
Do **not** commit your `.env` file.

### 🚀 Stretch goals (optional)
* **Become an API provider:** serve your screener results with [FastAPI](https://fastapi.tiangolo.com):
  `@app.get("/screen")` returning `my_picks.reset_index().to_dict("records")`, then call it with `curl`.
* **Go real-time:** subscribe to a crypto exchange's public WebSocket ticker feed with the `websockets`
  library and plot prices as they stream in.
* **Swap the universe:** screen the Dow 30, your favorite sector ETF holdings, or a list of crypto pairs (`BTC-USD`…).

In [ ]:
earned = sum(BADGES.values())
print(" ".join(k for k, v in BADGES.items() if v) or "No badges yet")
print(f"\nFinal tally: {earned}/{TOTAL_BADGES} badges" + ("  🎉 Quant desk certified!" if earned == TOTAL_BADGES else ""))